# Ingest CDS view -> Silver (ETP)
This notebook demonstrates reading a CDS view via JDBC, applying simple ETP (cleaning and type fixes), and writing Parquet files to the silver layer. Replace the placeholders with your connection details and run in a Fabric Spark notebook environment.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
spark = SparkSession.builder.getOrCreate()
print('Spark session ready:', spark)

In [ ]:
# JDBC connection placeholders - replace with your values or fetch from secrets store
jdbc_url = "jdbc:sqlserver://<HOST>;databaseName=<DB>;encrypt=true;trustServerCertificate=false;"
jdbc_table = "<CDS_VIEW_SCHEMA>.<CDS_VIEW_NAME>"
jdbc_props = {
    'user': '<DB_USER>',
    'password': '<DB_PASSWORD>'
}
# Example read (adjust format/driver for your connector)
df = spark.read.format('jdbc')
    .option('url', jdbc_url)
    .option('dbtable', jdbc_table)
    .option('user', jdbc_props['user'])
    .option('password', jdbc_props['password'])
    .load()
print('Read rows:', df.count())
display(df.limit(10))

In [ ]:
# Simple ETP: example cleaning and casting - adapt to your schema
# Drop fully-empty rows and trim string columns
from pyspark.sql.functions import trim
cols = df.columns
# Trim all string columns
for c in cols:
    if dict(df.dtypes)[c].startswith('string'):
        df = df.withColumn(c, trim(col(c)))
# Drop rows where all columns are null
df_clean = df.na.drop(how='all')
print('After cleaning rows:', df_clean.count())
display(df_clean.limit(10))

In [ ]:
# Write to silver layer as Parquet (replace silver_path)
silver_path = '<ONE_LAKE_ROOT>/fabric/silver/<table_name>'  # update from lakehouse_config.yml
df_clean.write.mode('overwrite').parquet(silver_path)
print('Wrote silver to', silver_path)
# Quick validation read
df_check = spark.read.parquet(silver_path)
print('Silver rows:', df_check.count())